# Lista 7

## scikit i Optuna

(6pkt + 3pkt)

Na liście znajduje się 5 zadań. Zadanie *optuna* jest za 2pkt, a inne zadania za 1pkt. Po rozwiązaniu zadania pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** — tylko wtedy przyznajemy punkty. Dodatkowo prześlij zadanie na platformie skos.

Zadania dodatkowe oznaczone są ⭐️. Za wykonanie każdego z nich otrzymasz 1pkt. 

## KFold

Wykonaj walidację krzyżową `KFold` (5 foldów) dla modelu `LinearRegression` na zbiorze California Housing.
Zapisz MSE dla każdego folda i podaj średnie MSE.

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

X, y = fetch_california_housing(return_X_y=True)

kf = KFold(n_splits=5)
model = LinearRegression()

mse_scores = []

for train_idx, test_idx in kf.split(X):
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict(X[test_idx])
    mse_scores.append(mean_squared_error(y[test_idx], preds))

print("MSE fold-by-fold:", mse_scores)
print("Średnie MSE:", np.mean(mse_scores))

MSE fold-by-fold: [0.48485856745731953, 0.6224973867349298, 0.6462104728578181, 0.5431995961545425, 0.4946848356387944]
Średnie MSE: 0.5582901717686809


## StratifiedKFold

Na zbiorze Iris zbuduj problem binarny: „Setosa” vs „inne”.
Porównaj proporcje klas w foldach między `KFold` a `StratifiedKFold`.
Przeprowadź walidację `LogisticRegression` (accuracy).

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

X, y_raw = load_iris(return_X_y=True)
y = (y_raw == 0).astype(int)

kf = KFold(n_splits=5)
skf = StratifiedKFold(n_splits=5)

print("Udział klasy pozytywnej (KFold):")
for _, test_idx in kf.split(X):
    print(y[test_idx].mean())

print("\nUdział klasy pozytywnej (StratifiedKFold):")
for _, test_idx in skf.split(X, y):
    print(y[test_idx].mean())

model = LogisticRegression(max_iter=200)
scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
print("\nAccuracy:", scores.mean())

Udział klasy pozytywnej (KFold):
1.0
0.6666666666666666
0.0
0.0
0.0

Udział klasy pozytywnej (StratifiedKFold):
0.3333333333333333
0.3333333333333333
0.3333333333333333
0.3333333333333333
0.3333333333333333

Accuracy: 1.0


## GridSearchCV

Wykonaj GridSearchCV dla `SVC`.
Siatka parametrów:

* C: [0.1, 1, 10]
* gamma: ["scale", "auto"]
* kernel: ["rbf", "poly"]

Użyj `StratifiedKFold(5)`. Wyświetl najlepsze parametry i wynik.

In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC

X, y = load_iris(return_X_y=True)

param_grid = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto"],
    "kernel": ["rbf", "poly"]
}

svm = SVC()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=skf,
    scoring='accuracy'
)

grid.fit(X, y)

print("Najlepsze parametry:", grid.best_params_)
print("Najlepszy wynik:", grid.best_score_)

Najlepsze parametry: {'C': 1, 'gamma': 'scale', 'kernel': 'poly'}
Najlepszy wynik: 0.9800000000000001


## RandomizedSearchCV

Wykonaj RandomizedSearchCV dla `RandomForestClassifier`.
Parametry losowane:

* n_estimators: 100–999
* max_depth: 2–39

30 losowań. Wypisz najlepsze parametry i accuracy.

In [5]:
from sklearn.datasets import load_iris
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

X, y = load_iris(return_X_y=True)

param_dist = {
    "n_estimators": randint(100, 999),
    "max_depth":    randint(2, 39)
}

model = RandomForestClassifier(random_state=42)

rnd = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=30,
    scoring='accuracy'
)

rnd.fit(X, y)

print("Najlepsze parametry:", rnd.best_params_)
print("Najlepszy wynik:", rnd.best_score_)

Najlepsze parametry: {'max_depth': 27, 'n_estimators': 365}
Najlepszy wynik: 0.9666666666666668


## Optuna

Twoim zadaniem jest przeprowadzić pełną optymalizację hiperparametrów modelu `GradientBoostingClassifier` z użyciem Optuny.

Wykonaj:

1. Wczytaj dane z `sklearn.datasets`.
2. Zdefiniuj funkcję celu, w której:

   * pobierzesz hiperparametry z `trial.suggest_*`,
   * stworzysz model `GradientBoostingClassifier`,
   * ocenisz go 5-krotną walidacją krzyżową,
   * zwrócisz średnią dokładność.
3. Uruchom optymalizację z `TPESampler` + `MedianPruner`, ustawiając parametry:

   * `direction="maximize"`
   * `storage="sqlite:///gb_optuna.db"`
   * `load_if_exists=True`
4. Uruchom drugą optymalizację z `RandomSampler` (tylko porównanie, bez zapisu do bazy).
5. Wygeneruj kilka (minimum 3) wykresy z `optuna.visualization` na podstawie study zapisanej w SQLite.
6. Uruchom Optuna Dashboard na bazie `gb_optuna.db`.
7. Wydrukuj porównanie wyników TPE vs RandomSampler.

In [2]:
import optuna
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.datasets import load_breast_cancer
from scipy.stats import randint, uniform

# ---------------------------------------------------------------
# 1) Wczytanie danych
# ---------------------------------------------------------------

X, y = load_breast_cancer(return_X_y=True)

# ---------------------------------------------------------------
# 2) Funkcja celu używana we wszystkich optymalizacjach
# ---------------------------------------------------------------

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 1, 6),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20)
    }

    model = GradientBoostingClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(estimator=model, X=X, y=y, cv=cv, scoring='accuracy')

    return scores.mean()

# ---------------------------------------------------------------
# 3) Główna optymalizacja — TPE + zapis do SQLite
# ---------------------------------------------------------------

study_sqlite = optuna.create_study(
    study_name="gb_sqlite_study",
    storage="sqlite:///gb_optuna.db",
    direction="maximize",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.MedianPruner()
)

study_sqlite.optimize(objective, n_trials=15, show_progress_bar=True)

print("Najlepsze parametry TPE:", study_sqlite.best_params)
print("Najlepsze accuracy TPE:", study_sqlite.best_value)

# ---------------------------------------------------------------
# 4) Druga optymalizacja — RandomSampler (bez zapisu do bazy)
# ---------------------------------------------------------------

study_random = optuna.create_study(direction="maximize", sampler=optuna.samplers.RandomSampler(seed=42))

study_random.optimize(objective, n_trials=15, show_progress_bar=True)

print("Najlepsze accuracy RandomSampler:", study_random.best_value)

# ---------------------------------------------------------------
# 5) Wizualizacje
# ---------------------------------------------------------------



[I 2025-11-25 14:25:26,414] Using an existing study with name 'gb_sqlite_study' instead of creating a new one.
Best trial: 15. Best value: 0.970144:   7%|▋         | 1/15 [00:08<02:00,  8.60s/it]

[I 2025-11-25 14:25:35,050] Trial 45 finished with value: 0.9666356155876417 and parameters: {'n_estimators': 245, 'learning_rate': 0.07190528055854216, 'max_depth': 5, 'subsample': 0.660959725874147, 'min_samples_split': 16}. Best is trial 15 with value: 0.9701443875174661.


Best trial: 15. Best value: 0.970144:  13%|█▎        | 2/15 [00:12<01:19,  6.11s/it]

[I 2025-11-25 14:25:39,419] Trial 46 finished with value: 0.9648812296227295 and parameters: {'n_estimators': 136, 'learning_rate': 0.151308086394156, 'max_depth': 4, 'subsample': 0.7693219121269644, 'min_samples_split': 14}. Best is trial 15 with value: 0.9701443875174661.


Best trial: 15. Best value: 0.970144:  20%|██        | 3/15 [00:22<01:31,  7.61s/it]

...

[I 2025-11-25 14:28:42,846] Trial 12 finished with value: 0.9631113181183046 and parameters: {'n_estimators': 186, 'learning_rate': 0.08869121921442981, 'max_depth': 5, 'subsample': 0.6783766633467947, 'min_samples_split': 7}. Best is trial 7 with value: 0.9683434249340165.


Best trial: 7. Best value: 0.968343:  93%|█████████▎| 14/15 [01:12<00:05,  5.94s/it]

[I 2025-11-25 14:28:49,245] Trial 13 finished with value: 0.9613569321533924 and parameters: {'n_estimators': 240, 'learning_rate': 0.050868025242681164, 'max_depth': 5, 'subsample': 0.5372753218398854, 'min_samples_split': 20}. Best is trial 7 with value: 0.9683434249340165.


Best trial: 7. Best value: 0.968343: 100%|██████████| 15/15 [01:15<00:00,  5.06s/it]

[I 2025-11-25 14:28:53,129] Trial 14 finished with value: 0.964834653004192 and parameters: {'n_estimators': 321, 'learning_rate': 0.06762754764491, 'max_depth': 1, 'subsample': 0.9077307142274171, 'min_samples_split': 15}. Best is trial 7 with value: 0.9683434249340165.
Najlepsze accuracy RandomSampler: 0.9683434249340165


In [8]:
from optuna.visualization import (
    plot_optimization_history,
    plot_parallel_coordinate,
    plot_slice,
    plot_contour,
    plot_param_importances,
    plot_intermediate_values,
    plot_edf,
    plot_timeline,
    plot_rank,
)

plot_optimization_history(study_sqlite).show()
plot_param_importances(study_sqlite).show()
plot_parallel_coordinate(study_sqlite).show()

# ---------------------------------------------------------------
# 6) Dashboard — komenda do uruchomienia w terminalu:
# ---------------------------------------------------------------
# optuna-dashboard sqlite:///gb_optuna.db

# ---------------------------------------------------------------
# 7) Porównanie wyników
# ---------------------------------------------------------------

print("===============================================")
print("PORÓWNANIE")
print("TPE (SQLite):   ", study_sqlite.best_value)
print("RandomSampler:   ", study_random.best_value)
print("===============================================")

PORÓWNANIE
TPE (SQLite):    0.9701443875174661
RandomSampler:    0.9683434249340165


## ⭐️ GroupKFold

Wygeneruj zbiór danych składający się z **500 próbek** oraz **10 cech**.
Następnie przydziel każdą próbkę do jednej z **50 grup**, tak aby każda grupa zawierała dokładnie **10 próbek**.

Celem zadania jest sprawdzenie, jak działa walidacja krzyżowa z podziałem na grupy.
Użyj **GroupKFold(5)** – oznacza to, że dane będą dzielone tak, aby **żadne próbki z tej samej grupy nie trafiły jednocześnie do zbioru treningowego i testowego**.

Na tak przygotowanych danych wykonaj walidację krzyżową modelu **RandomForestClassifier**, a następnie wypisz średnią dokładność.

In [1]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score
import numpy as np

# 1) Generowanie sztucznych danych
X, y = make_classification(n_samples=500, n_features=10, random_state=42)

# 2) Przydział 500 próbek do 50 grup (po 10 próbek w każdej)
groups = np.repeat(np.arange(50), 10)

model = RandomForestClassifier(random_state=42)

gkf = GroupKFold(n_splits=50)
scores = []

fold = 1

for train_idx, val_idx in gkf.split(X, y, groups=groups):
    train_groups = np.unique(groups[train_idx])
    val_groups = np.unique(groups[val_idx])

    model.fit(X[train_idx], y[train_idx])
    scores.append(model.score(X[val_idx], y[val_idx]))

    # print(f"\n=== Fold {fold} ===")
    # print("Train groups:", train_groups)
    # print("Val groups:  ", val_groups)

    # intersection = np.intersect1d(train_groups, val_groups)
    # print("Intersection (powinno być puste):", intersection)

    fold += 1

print("\nGroupKFold accuracy scores:", scores)
print("Average accuracy:", np.mean(scores))



GroupKFold accuracy scores: [1.0, 0.9, 1.0, 0.9, 0.8, 1.0, 1.0, 1.0, 1.0, 0.9, 1.0, 1.0, 0.9, 1.0, 0.9, 1.0, 0.8, 1.0, 1.0, 0.9, 0.9, 0.8, 1.0, 0.9, 1.0, 1.0, 1.0, 0.9, 0.8, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9, 1.0, 0.9, 1.0, 0.9, 0.9, 1.0, 0.9, 1.0, 1.0, 1.0, 1.0, 0.9, 0.8]
Average accuracy: 0.946


## ⭐️ TimeSeriesSplit

Wygeneruj **sztuczny szereg czasowy** składający się z 1000 obserwacji.
Przyjmij, że cechą wejściową jest indeks czasu (0,1,2,…,999), a wartością docelową sygnał sinusoidalny z zakłóceniem (szumem gaussowskim).

Następnie zastosuj **TimeSeriesSplit(5)**, który wykonuje walidację krzyżową dostosowaną do danych sekwencyjnych — każdy kolejny fold używa coraz większej części historii jako zbioru treningowego.

Na każdym foldzie wytrenuj model **GradientBoostingRegressor**, następnie policz i wypisz **MAE (Mean Absolute Error)** na zbiorze testowym.

Wynik powinien zawierać wartość MAE dla każdego folda osobno.

In [3]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# 1) Generowanie sztucznego szeregu czasowego
np.random.seed(42)
X = np.arange(1000).reshape(-1, 1)
y = np.sin(X[:, 0] / 20) + np.random.normal(scale=0.1, size=1000)

tscv = TimeSeriesSplit(n_splits=5)
fold = 0
mae_scores = []

model = GradientBoostingRegressor()
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    print(f"Fold {fold}: {mae}")
    fold+=1



Fold 0: 0.8288663064508833
Fold 1: 0.7330599469306063
Fold 2: 0.6298690477081538
Fold 3: 0.9575344251525488
Fold 4: 0.651871473738235


## ⭐️ Optymalizacja hiperparametrów SVM z użyciem Optuny

Twoim zadaniem jest zoptymalizować hiperparametry modelu **Support Vector Machine (SVC)** z wykorzystaniem biblioteki **Optuna**.

Wykonaj następujące kroki:

1. Wczytaj zbiór danych **Wine** ze `sklearn.datasets`.
2. Podziel dane na zbiór treningowy i testowy (np. 80/20).
3. Zdefiniuj funkcję celu `objective(trial)`, która:

   * wybierze hiperparametry `C` oraz `gamma` (oba w skali logarytmicznej),
   * wybierze kernel spośród: `"rbf"`, `"poly"`, `"sigmoid"`,
   * utworzy model `SVC` z dobranymi parametrami,
   * wykona walidację krzyżową (5-fold),
   * zwróci średnią dokładność walidacyjną jako wartość celu.
4. Utwórz study używając `TPESampler`.
5. Uruchom optymalizację na 40 próbach.
6. Wypisz najlepsze hiperparametry oraz najlepszy uzyskany wynik.
7. Wytrenuj finalny model na pełnym zbiorze treningowym z najlepszymi hiperparametrami i oceń go na zbiorze testowym — wypisz finalny accuracy.

Wyniki powinny pokazać:

* najlepsze hiperparametry wybrane przez Optunę,
* wynik walidacji,
* wynik na zbiorze testowym.

In [5]:
import optuna
from sklearn.svm import SVC
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

# 1) Wczytanie danych Wine
X, y = load_wine(return_X_y=True)

# 2) Podział danych (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

# 3) Funkcja celu Optuny
def objective(trial):
    params = {
        "kernel": trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"]),
        "C": trial.suggest_float("C", 1e-5, 1e-1, log=True),
        "gamma": trial.suggest_float("gamma", 1e-5, 1e-1, log=True)
    }
    model = SVC(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(estimator=model, X=X, y=y, cv=cv, scoring='accuracy')

    return scores.mean()

# 4) Utworzenie study
study = optuna.create_study(sampler=optuna.samplers.TPESampler())

# 5) Optymalizacja (40 prób)
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Najlepsze parametry:", study.best_params)
print("Najlepszy wynik CV:", study.best_value)

# 6) Trening finalnego modelu na najlepszych parametrach
model = SVC(kernel=study.best_params["kernel"], C=study.best_params["C"], gamma=study.best_params["gamma"])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy w modelu na najlepszych parametrach: {acc}")

# 7) Ocena na zbiorze testowym
# Uzupełnij

[I 2025-11-28 13:16:30,638] A new study created in memory with name: no-name-557b3b24-0534-49f3-950d-e73e0094faf5
Best trial: 0. Best value: 0.399048:   5%|▌         | 2/40 [00:00<00:02, 17.61it/s]

[I 2025-11-28 13:16:30,777] Trial 0 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 0.0018755450512837537, 'gamma': 0.0007346996591435186}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:30,826] Trial 1 finished with value: 0.6801587301587302 and parameters: {'kernel': 'poly', 'C': 0.002265444546438787, 'gamma': 1.7187406355173543e-05}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:30,916] Trial 2 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 0.018161125081366712, 'gamma': 0.0003956165852919087}. Best is trial 0 with value: 0.3990476190476191.


Best trial: 0. Best value: 0.399048:  15%|█▌        | 6/40 [00:00<00:03,  9.15it/s]

[I 2025-11-28 13:16:31,209] Trial 3 finished with value: 0.9382539682539681 and parameters: {'kernel': 'poly', 'C': 4.0278852779136636e-05, 'gamma': 0.0007349293186669837}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:31,303] Trial 4 finished with value: 0.3990476190476191 and parameters: {'kernel': 'sigmoid', 'C': 0.0005283479235693297, 'gamma': 0.0001362705423402837}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:31,373] Trial 5 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 0.0001258723386377091, 'gamma': 3.824860413879425e-05}. Best is trial 0 with value: 0.3990476190476191.


Best trial: 0. Best value: 0.399048:  20%|██        | 8/40 [00:00<00:03, 10.32it/s]

...

[I 2025-11-28 13:16:34,190] Trial 30 finished with value: 0.9382539682539681 and parameters: {'kernel': 'poly', 'C': 0.0018107766147992166, 'gamma': 0.00020324416903741614}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:34,303] Trial 31 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 0.00010203344335401728, 'gamma': 2.5268802796580324e-05}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:34,345] Trial 32 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 0.00010326382713243258, 'gamma': 1.423146691400634e-05}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:34,385] Trial 33 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 4.451273467999237e-05, 'gamma': 2.727457618036874e-05}. Best is trial 0 with value: 0.3990476190476191.


Best trial: 0. Best value: 0.399048:  95%|█████████▌| 38/40 [00:03<00:00, 14.47it/s]

[I 2025-11-28 13:16:34,444] Trial 34 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 2.0389034477851293e-05, 'gamma': 5.2036751506005945e-05}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:34,502] Trial 35 finished with value: 0.7192063492063492 and parameters: {'kernel': 'poly', 'C': 0.0003586080992955766, 'gamma': 0.00010737490594133652}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:34,544] Trial 36 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 4.62544340188343e-05, 'gamma': 0.00044277743127188864}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:34,593] Trial 37 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C': 1.0854170870714013e-05, 'gamma': 0.0009028745866427989}. Best is trial 0 with value: 0.3990476190476191.
[I 2025-11-28 13:16:34,637] Trial 38 finished with value: 0.3990476190476191 and parameters: {'kernel': 'rbf', 'C':

Best trial: 0. Best value: 0.399048: 100%|██████████| 40/40 [00:04<00:00,  8.02it/s]

[I 2025-11-28 13:16:35,704] Trial 39 finished with value: 0.9661904761904762 and parameters: {'kernel': 'poly', 'C': 0.03798208473844055, 'gamma': 0.003483626050205459}. Best is trial 0 with value: 0.3990476190476191.
Najlepsze parametry: {'kernel': 'rbf', 'C': 0.0018755450512837537, 'gamma': 0.0007346996591435186}
Najlepszy wynik CV: 0.3990476190476191
Accuracy w modelu na najlepszych parametrach: 0.3888888888888889
